# Cancel In-Progress Fabric Jobs

Companion notebook for [`cancel_in_progress_jobs.py`](./cancel_in_progress_jobs.py).

**Inside a Fabric notebook** the script authenticates automatically via
`notebookutils.mssparkutils.credentials.getToken(...)` - no `az login` needed.

Outside Fabric (local Jupyter) the script falls back to `azure-identity`
or `az` CLI.

## Load the helper

In [ ]:
from cancel_in_progress_jobs import cancel_in_progress_jobs

## RECOMMENDED for Fabric Admins: tenant-wide via Activity Events

One API call returns every running notebook + pipeline run across the
entire tenant - no per-workspace fanout, no throttling.

**Requires Fabric Administrator role.**

In [ ]:
# Preview only
cancel_in_progress_jobs(use_activity_events=True, dry_run=True)

In [ ]:
# Cancel and verify (poll=60 waits up to 60s per cancelled job)
cancel_in_progress_jobs(
    use_activity_events=True,
    activity_lookback=60,                          # minutes
    exclude_workspace=["Admin", "Fabric Admin"],   # never touch these
    poll=60,
)

In [ ]:
# Continuous monitoring for 30 minutes
cancel_in_progress_jobs(
    use_activity_events=True,
    loop=True, loop_duration=30, poll_interval=30,
    exclude_workspace=["Admin", "Fabric Admin"],
    sleep_interval=0.1,
    poll=45,
)

## Targeted to specific workspaces (no admin role needed)

For users without Fabric Admin role, name the workspaces explicitly.

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",  # name OR GUID; list OK
    dry_run=True,
)

In [ ]:
cancel_in_progress_jobs(
    workspace=[
        "crestshield-smartclaims-sachinsaraf",
        "crestshield-smartclaims-secondary",
    ],
    poll=60,
)

## Targeting specific items

All filters can be combined.

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",
    item=["01_Bronze*", "*Silver*"],     # name or GUID, wildcards/substring ok
    item_type=["Notebook", "DataPipeline"],
    exclude_item=["*Production*"],
    poll=60,
)

## Also stop live notebook Spark sessions

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",
    stop_notebook_sessions=True,
    poll=60,
)

## Safety gate

Calling `cancel_in_progress_jobs()` with no arguments will try to scan
every workspace the user can see. If that's more than 10 (default), it
refuses to proceed and tells you which alternatives to use.

In [ ]:
# This returns {'error': 'too_many_workspaces', 'workspace_count': N}
# if the user has access to more than 10 workspaces.
cancel_in_progress_jobs(dry_run=True)

# To override (slow + may hit 429s):
# cancel_in_progress_jobs(dry_run=True, confirm_large_scan=True,
#                         max_default_workspaces=200, sleep_interval=0.5)